# 1.1.Process downloaded loaddata to match local image data
This notebook:
1. Reads in the previously downloaded loaddata file.
2. Cross-compare loaddata file paths with local data downloads, correct paths with best effort. 
3. Write loaddata files with fixed path.

In [1]:
from pathlib import Path, PurePosixPath

import pandas as pd

from utils.validate_config import (
    load_yaml_config,
    require_config_directory,
    require_config_string,
    require_config_value,
)

## Pathing

In [2]:
config = load_yaml_config("degradation_config.yaml")
local_image_dir = require_config_directory(config, "local_image_dir").resolve()
channels = require_config_value(config, "channels")

metadata_download_path = Path("metadata")
metadata_download_path.mkdir(parents=True, exist_ok=True)

loaddata_dir = metadata_download_path / "loaddatas"
loaddata_files = sorted(
    path for path in loaddata_dir.glob("*.csv") if not path.name.endswith(".fixed.csv")
)
if not loaddata_files:
    raise FileNotFoundError(f"No source loaddata CSV files found in {loaddata_dir}.")

## Fixing loaddata paths
Assumes local image data exists as the same file structure as that being processed in pediatric cancer atlas profiling repo.  

In [3]:
path_col_prefix = require_config_string(config, "path_col_prefix")
file_col_prefix = require_config_string(config, "file_col_prefix")
source_path_prefix = PurePosixPath(
    require_config_string(config, "source_path_prefix")
)

path_columns = {channel: f"{path_col_prefix}{channel}" for channel in channels}
file_columns = {channel: f"{file_col_prefix}{channel}" for channel in channels}

# Validate every input schema before writing any output, so a malformed plate fails early.
for loaddata_file in loaddata_files:
    csv_columns = set(pd.read_csv(loaddata_file, nrows=0).columns)
    missing_columns_by_channel = {
        channel: [
            column
            for column in (path_columns[channel], file_columns[channel])
            if column not in csv_columns
        ]
        for channel in channels
    }
    missing_columns_by_channel = {
        channel: columns
        for channel, columns in missing_columns_by_channel.items()
        if columns
    }
    if missing_columns_by_channel:
        missing_details = "; ".join(
            f"{channel}: {', '.join(columns)}"
            for channel, columns in missing_columns_by_channel.items()
        )
        raise KeyError(
            f"Missing anticipated columns in {loaddata_file.name}: {missing_details}"
        )

# Process each loaddata CSV file to validate and fix paths and filenames.
for loaddata_file in loaddata_files:
    print(f"Processing loaddata file: {loaddata_file}")

    loaddata_df = pd.read_csv(loaddata_file)
    print(f"Loaded loaddata file with shape: {loaddata_df.shape}")

    # Collect failed path/file counts
    failure_report = pd.DataFrame(
        0, index=pd.Index(channels, name="channel"), columns=["path", "file"], dtype=int
    )

    # Remap acquisition-machine directories to the local data root and reject missing paths.
    for channel in channels:
        path_col = path_columns[channel]
        fixed_paths = []

        for path_cell_content in loaddata_df[path_col]:
            if pd.isna(path_cell_content):
                fixed_paths.append(None)
                failure_report.at[channel, "path"] += 1
                continue

            source_path = PurePosixPath(str(path_cell_content))
            try:
                relative_path = source_path.relative_to(source_path_prefix)
            except ValueError:
                fixed_paths.append(None)
                failure_report.at[channel, "path"] += 1
                continue

            fixed_path = local_image_dir.joinpath(*relative_path.parts)
            fixed_paths.append(str(fixed_path))

        loaddata_df[path_col] = fixed_paths

    # Remove invalid directories first; only surviving rows can form meaningful TIFF paths.
    rows_before_path_drop = len(loaddata_df)
    loaddata_df = loaddata_df.dropna(subset=list(path_columns.values())).copy()
    print(f"Dropped {rows_before_path_drop - len(loaddata_df)} rows with missing paths")

    # Pair each channel's validated directory and filename to detect missing TIFF transfers.
    all_channel_files_exist = pd.Series(True, index=loaddata_df.index)
    for channel in channels:
        path_col = path_columns[channel]
        file_col = file_columns[channel]
        channel_files_exist = pd.Series(False, index=loaddata_df.index)

        for row_index, (path_cell_content, file_cell_content) in loaddata_df[
            [path_col, file_col]
        ].iterrows():
            if pd.isna(file_cell_content):
                continue

            fixed_file = Path(path_cell_content) / str(file_cell_content)
            channel_files_exist.at[row_index] = fixed_file.is_file()

        failure_report.at[channel, "file"] = int((~channel_files_exist).sum())
        all_channel_files_exist &= channel_files_exist

    # Keep only rows that are complete across every anticipated imaging channel.
    rows_before_file_drop = len(loaddata_df)
    loaddata_df = loaddata_df.loc[all_channel_files_exist].copy()
    print(f"Dropped {rows_before_file_drop - len(loaddata_df)} rows with missing TIFF files")
    print("\nFailed validation counts by channel:")
    print(failure_report.to_string())

    output_file = loaddata_file.with_name(f"{loaddata_file.stem}.fixed.parquet")
    loaddata_df.to_parquet(output_file, index=False)
    print(f"Saved {len(loaddata_df)} fully validated rows to {output_file}\n")

Processing loaddata file: metadata/loaddatas/BR00143976_concatenated.csv
Loaded loaddata file with shape: (2160, 25)
Dropped 0 rows with missing paths
Dropped 0 rows with missing TIFF files

Failed validation counts by channel:
                 path  file
channel                    
OrigER              0     0
OrigRNA             0     0
OrigAGP             0     0
OrigMito            0     0
OrigDNA             0     0
OrigBrightfield     0     0
Saved 2160 fully validated rows to metadata/loaddatas/BR00143976_concatenated.fixed.parquet

Processing loaddata file: metadata/loaddatas/BR00143977_concatenated.csv
Loaded loaddata file with shape: (1259, 25)
Dropped 0 rows with missing paths
Dropped 0 rows with missing TIFF files

Failed validation counts by channel:
                 path  file
channel                    
OrigER              0     0
OrigRNA             0     0
OrigAGP             0     0
OrigMito            0     0
OrigDNA             0     0
OrigBrightfield     0     0
Sav